In [ ]:
import pickle
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data_utils
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import defaultdict

import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.sparse import issparse
import inspect
import json
import random 
from tqdm import tqdm
from collections import Counter
from pathlib import Path
from typing import Optional, Dict, List, Tuple
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ============================================= function split data =============================================
class TestSplitter(object):
    def __init__(self, args):
        self.test_size = args['test_size']
        self.uid = 'user_id'
        self.tid = 'item_id'

    def split(self, df):
        train_index, test_index = split_test(df, self.test_size, self.uid)

        return train_index, test_index


class ValidationSplitter(object):
    def __init__(self, args):
        # self.fold_num = args.fold_num
        self.val_size = args['val_size']
        self.uid = 'user_id'
        self.tid = 'item_id'

    def split(self, df):
        train_val_index_zip = split_validation(df, self.val_size, self.uid)

        return train_val_index_zip


def split_test(df, test_size=0.1, uid='user_id'):

    test_ids = df.groupby(uid).apply(
        lambda x: x.sample(frac=test_size).index
    )
    test_ids = test_ids.explode().dropna().values.astype(int)
    # test_ids = np.array([int(x) for x in test_ids if not pd.isna(x)])
    test_ids = np.array(list(test_ids))
    train_ids = np.setdiff1d(df.index.values, test_ids)

    return train_ids, test_ids


def split_validation(train_set, val_size=.1, uid='user_id'):

    train_set = train_set.reset_index(drop=True)

    # train_set_list, val_set_list = [], []
    # for _ in range(fold_num):
    val_ids = train_set.groupby(uid).apply(
        lambda x: x.sample(frac=val_size).index
    )
    val_ids = val_ids.explode().dropna().values.astype(int)
    # val_ids = np.array([int(x) for x in val_ids if not pd.isna(x)])
    # val_ids = np.array(list(val_ids))
    train_ids = np.setdiff1d(train_set.index.values, val_ids)

    # train_set     _list.append(train_ids)
    # val_set_list.append(val_ids)

    return train_ids, val_ids 

# ============================================= function metrics =============================================

class Metric(object):
    def __init__(self, config) -> None:
        self.metrics = config['metrics']
        self.item_num = config['item_num']
        self.item_pop = config['item_pop'] if 'coverage' in self.metrics else None
        self.i_categories = config['i_categories'] if 'diversity' in self.metrics else None

    def run(self, test_ur, pred_ur, test_u):
        res = []
        for mc in self.metrics:
            if mc == 'ndcg':
                kpi = NDCG(test_ur, pred_ur, test_u)
            elif mc == 'recall':
                kpi = Recall(test_ur, pred_ur, test_u)
            elif mc == 'precision':
                kpi = Precision(test_ur, pred_ur, test_u)
            else:
                raise ValueError(f'Invalid metric name {mc}')

            res.append(kpi)

        return res

def Precision(test_ur, pred_ur, test_u):
    res = []
    for idx in range(len(test_u)):
        u = test_u[idx]
        gt = test_ur[u]
        pred = pred_ur[idx]
        pre = np.isin(pred, list(gt)).sum() / len(pred)

        res.append(pre)

    return np.mean(res)


def Recall(test_ur, pred_ur, test_u):
    res = []
    for idx in range(len(test_u)):
        u = test_u[idx]
        gt = test_ur[u]
        pred = pred_ur[idx]
        rec = np.isin(pred, list(gt)).sum() / len(gt)

        res.append(rec)

    return np.mean(res)

def getDCG(scores):
    return np.sum(
        np.divide(np.power(2, scores) - 1, np.log2(np.arange(scores.shape[0], dtype=np.float32) + 1)+1),
        # np.divide(scores, np.log2(np.arange(scores.shape[0], dtype=np.float32) + 2)+1),
        dtype=np.float32)

def getNDCG(rank_list, pos_items):
    relevance = np.ones_like(pos_items)
    it2rel = {it: r for it, r in zip(pos_items, relevance)}
    rank_scores = np.asarray([it2rel.get(it, 0.0) for it in rank_list], dtype=np.float32)
    idcg = getDCG(relevance)

    dcg = getDCG(rank_scores)

    if dcg == 0.0:
        return 0.0

    ndcg = dcg / idcg
    return ndcg

def NDCG(test_ur, pred_ur, test_u):
    res = []
    for idx in range(len(test_u)):
        u = test_u[idx]
        gt = test_ur[u]
        pred = pred_ur[idx]
        nd = getNDCG(pred, gt)
        res.append(nd)
    return np.mean(res)


def AUC(test_ur, pred_ur, test_u):
    res = []

    for idx in range(len(test_u)):
        u = test_u[idx]
        gt = test_ur[u]
        pred = pred_ur[idx]

        r = np.isin(pred, list(gt))
        pos_num = r.sum()
        neg_num = len(pred) - pos_num

        # Handle edge cases: if no positive or no negative items, AUC is undefined
        if pos_num == 0 or neg_num == 0:
            continue  # Skip this user instead of adding NaN

        pos_rank_num = 0
        for j in range(len(r) - 1):
            if r[j]:
                pos_rank_num += np.sum(~r[j + 1:])

        auc = pos_rank_num / (pos_num * neg_num)
        res.append(auc)

    return np.mean(res) if len(res) > 0 else 0.0

def AUC_true_neg(test_ur, pred_scores, true_neg_dict, test_u):
    aucs = []
    for idx, u in enumerate(test_u):
        pos = set(test_ur[u])
        neg = true_neg_dict.get(u, set())
        if len(pos) == 0 or len(neg) == 0:
            continue

        scores = pred_scores[idx]
        pos_scores = [scores[i] for i in pos if i < len(scores)]
        neg_scores = [scores[i] for i in neg if i < len(scores)]

        cnt = 0
        for ps in pos_scores:
            cnt += sum(ps > ns for ns in neg_scores)

        aucs.append(cnt / (len(pos_scores) * len(neg_scores)))
    return np.mean(aucs) if aucs else 0.0

# ============================================= function get data =============================================

def get_ur(df):
    print("Method of getting user-rating pairs")
    ur = df.groupby('user_id').item_id.apply(list).to_dict()
    # print(ur)
    return ur

class BasicDataset(data_utils.Dataset):
    def __init__(self, samples):
  
        super(BasicDataset, self).__init__()
        self.data = samples

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index][0], self.data[index][1], self.data[index][2]

class Cf_valDataset(data_utils.Dataset):
    def __init__(self, data):
        super(Cf_valDataset, self).__init__()
        self.user = data
        # self.data = data

    def __len__(self):
        return len(self.user)

    def __getitem__(self, index):
        user = self.user[index]
        return torch.tensor(user)#, torch.tensor(self.data[user])

def get_train_loader(dataset, args):
    dataloader = data_utils.DataLoader(dataset, batch_size=args['train_batch_size'], shuffle=True, pin_memory=True)
    return dataloader

def get_val_loader(dataset, args):
    dataloader = data_utils.DataLoader(dataset, batch_size=args['val_batch_size'], shuffle=False, pin_memory=True)
    return dataloader

def get_test_loader(dataset, args):
    dataloader = data_utils.DataLoader(dataset, batch_size=args['test_batch_size'], shuffle=False, pin_memory=True)
    return dataloader

def get_inter_matrix(df, args):
    '''
    get the whole sparse interaction matrix
    '''
    print("get the whole sparse interaction matrix")
    user_num, item_num = args['user_num'], args['item_num']

    src, tar = df['user_id'].values, df['item_id'].values
    data = df['click'].values

    mat = sp.coo_matrix((data, (src, tar)), shape=(user_num, item_num))

    return mat

def build_relation_matrices_from_df(df: pd.DataFrame, relations: List[str], user_num: int, item_num: int) -> Dict[str, sp.coo_matrix]:
    """
    df: should have columns ['user_id', 'item_id', <relations...>] with binary flags 0/1
    relations: e.g. ['click','like','share','follow','exposed']
    Returns dict: relation -> scipy.sparse.coo_matrix (user_num x item_num)
    """
    rel_mats = {}
    for r in relations:
        sub = df[df[r] == 1]
        if sub.shape[0] == 0:
            rel_mats[r] = sp.coo_matrix((user_num, item_num))
            continue
        rows = sub['user_id'].values.astype(np.int32)
        cols = sub['item_id'].values.astype(np.int32)
        data = np.ones(len(rows), dtype=np.float32)
        mat = sp.coo_matrix((data, (rows, cols)), shape=(user_num, item_num))
        rel_mats[r] = mat
    return rel_mats

# ============================================= function neg sampler =============================================
import numpy as np
from collections import Counter

class HybridNegativeSampler:
    """
    Multi-method negative sampler for implicit CF / BPR triples.

    Supported methods:
      - 'uniform'  : sample unobserved items uniformly
      - 'high-pop' : sample unobserved items by popularity (freq^(3/4))
      - 'low-pop'  : sample unobserved items by inverse popularity
      - 'true_neg' : sample from provided true negative pool per user
      - 'hybrid'   : mix (true_neg + uniform) with ratio = sample_ratio

    Usage:
      sampler = HybridNegativeSampler(cf_args)
      triples = sampler.sampling(df_train_edges, train_ur, true_neg_df=final_neg_pool)
    """

    def __init__(self, args: dict):
        self.user_num = args['user_num']
        self.item_num = args['item_num']
        self.num_ng = args.get('num_ng', 4)

        self.method = args.get('sampler_method', 'uniform')
        self.sample_ratio = float(args.get('sample_ratio', 0.5))  # used by hybrid
        self.seed = int(args.get('seed', 42))

        assert self.method in ['uniform', 'high-pop', 'low-pop', 'true_neg', 'hybrid'], \
            f"Invalid sampler_method: {self.method}"

        np.random.seed(self.seed)

        # will be built if needed
        self.pop_prob = None  # size = item_num

    # ---------- popularity distribution ----------
    def _build_pop_prob(self, train_edges_df):
        """
        train_edges_df: DataFrame with columns ['user_id','item_id'] (encoded)
        """
        cnt = Counter(train_edges_df['item_id'].values.tolist())
        freq = np.zeros(self.item_num, dtype=np.float64)
        for i, c in cnt.items():
            if 0 <= i < self.item_num:
                freq[i] = c

        # smoothing like word2vec negative sampling: p(i) ~ f(i)^(3/4)
        prob = freq / (freq.sum() + 1e-12)
        prob = np.power(prob, 0.75)

        if self.method == 'high-pop':
            prob = prob / (prob.sum() + 1e-12)
        elif self.method == 'low-pop':
            # inverse popularity: items with low freq get higher probability
            inv = 1.0 - (prob / (prob.max() + 1e-12))
            inv = np.clip(inv, 0.0, None)
            prob = inv / (inv.sum() + 1e-12)
        else:
            prob = prob / (prob.sum() + 1e-12)

        self.pop_prob = prob.astype(np.float64)

    # ---------- true negative dict ----------
    @staticmethod
    def _build_true_neg_dict(true_neg_df):
        """
        true_neg_df: DataFrame ['user_id','item_id'] encoded, TRAIN-ONLY (no leakage)
        """
        d = {}
        if true_neg_df is None or len(true_neg_df) == 0:
            return d
        for u, i in zip(true_neg_df['user_id'].values, true_neg_df['item_id'].values):
            d.setdefault(int(u), set()).add(int(i))
        return d

    # ---------- single draw helpers ----------
    def _sample_uniform_one(self, u_seen, chosen):
        j = np.random.randint(self.item_num)
        while (j in u_seen) or (j in chosen):
            j = np.random.randint(self.item_num)
        return j

    def _sample_pop_one(self, u_seen, chosen):
        # assumes self.pop_prob built
        j = int(np.random.choice(self.item_num, p=self.pop_prob))
        tries = 0
        while (j in u_seen) or (j in chosen):
            j = int(np.random.choice(self.item_num, p=self.pop_prob))
            tries += 1
            if tries > 50:
                # fallback to uniform to avoid dead-loop for dense users
                return self._sample_uniform_one(u_seen, chosen)
        return j

    def _sample_true_neg_one(self, u, u_seen, chosen, true_neg_dict):
        pool = list(true_neg_dict.get(u, []))
        if len(pool) == 0:
            return None
        j = int(np.random.choice(pool))
        tries = 0
        while (j in u_seen) or (j in chosen):
            j = int(np.random.choice(pool))
            tries += 1
            if tries > 50:
                return None
        return j

    # ---------- main API ----------
    def sampling(self, train_edges_df, train_ur, true_neg_df=None):
        """
        Returns: np.ndarray of shape [num_triples, 3] with columns [u, pos, neg]
        Inputs:
          - train_edges_df: df containing positive edges used to build popularity dist (encoded)
          - train_ur: dict user -> list(pos_items) (encoded)
          - true_neg_df: df containing (u,i) true negatives, TRAIN-ONLY (encoded)
        """
        if self.num_ng <= 0:
            raise ValueError("num_ng must be > 0 for BPR")

        # Build pop dist only if needed
        if self.method in ['high-pop', 'low-pop']:
            self._build_pop_prob(train_edges_df)

        true_neg_dict = self._build_true_neg_dict(true_neg_df) if self.method in ['true_neg', 'hybrid'] else {}

        triples = []
        for u in range(self.user_num):
            pos_list = train_ur.get(u, [])
            if len(pos_list) == 0:
                continue
            u_seen = set(pos_list)

            for pos in pos_list:
                chosen = set()

                # --- HYBRID: mix true_neg + uniform (or you can change to pop if you want) ---
                if self.method == 'hybrid':
                    k_true = int(round(self.num_ng * self.sample_ratio))
                    k_true = max(0, min(self.num_ng, k_true))
                    k_other = self.num_ng - k_true

                    # true neg part
                    for _ in range(k_true):
                        j = self._sample_true_neg_one(u, u_seen, chosen, true_neg_dict)
                        if j is None:
                            break
                        chosen.add(j)

                    # uniform part (fallback)
                    while len(chosen) < self.num_ng:
                        chosen.add(self._sample_uniform_one(u_seen, chosen))

                elif self.method == 'true_neg':
                    # only true negatives, fallback to uniform if not enough
                    while len(chosen) < self.num_ng:
                        j = self._sample_true_neg_one(u, u_seen, chosen, true_neg_dict)
                        if j is None:
                            # fallback uniform to guarantee enough negs
                            j = self._sample_uniform_one(u_seen, chosen)
                        chosen.add(j)

                elif self.method == 'uniform':
                    while len(chosen) < self.num_ng:
                        chosen.add(self._sample_uniform_one(u_seen, chosen))

                elif self.method in ['high-pop', 'low-pop']:
                    while len(chosen) < self.num_ng:
                        chosen.add(self._sample_pop_one(u_seen, chosen))

                else:
                    raise ValueError(f"Unknown method: {self.method}")

                for neg in chosen:
                    triples.append([u, int(pos), int(neg)])

        return np.asarray(triples, dtype=np.int32)

# ============================================= Model =============================================


class LightGCN(nn.Module):
    """A self-contained LightGCN implementation.

    Features:
    - Builds normalized adjacency from a scipy COO interaction matrix (user-item bipartite)
    - Layer-wise propagation (no non-linearities / no feature transform)
    - Mean aggregation over (L+1) layers (including the 0-th embedding)
    - Supports BPR, hinge (HL), TOP1 (TL), and point-wise (BCEWithLogits, MSE) losses via configure_loss
    - Ranking utilities: rank(), full_rank()
    """
    def __init__(self, args):
        super().__init__()
        self.num_users = args['user_num']
        self.num_items = args['item_num']
        self.embedding_dim = args.get('embedding_dim', 64)
        self.num_layers = args.get('num_layers', 3)
        self.interaction_matrix = args.get('interaction_matrix', None)
        self.device = torch.device(args.get('device', 'cpu'))
        self.reg_1 = args.get('reg_1', 0.0)
        self.reg_2 = args.get('reg_2', 0.0)
        self.dropout = nn.Dropout(args.get('dropout', 0.0))
        self.lr = args.get('lr', 0.001)
        self.topk = args.get('k', 20)
        self.val_ur = args.get('val_ur', None)
        self.val_u = args.get('val_u', None)
        self.early_stop = args.get('early_stop', True)
        self.save_path = args.get('save_path', './')
        self.true_neg = args.get('true_neg', False)
        self.true_neg_dict = args.get('true_neg_dict', {})
        self.load = args.get('load', False)


        # storage variables for rank evaluation acceleration
        self.restore_user_e = None
        self.restore_item_e = None

        # Embeddings
        self.embed_user = nn.Embedding(self.num_users, self.embedding_dim)
        self.embed_item = nn.Embedding(self.num_items, self.embedding_dim)
        self.apply(self._init_weights)

        if self.interaction_matrix is None:
            raise ValueError("interaction_matrix (scipy sparse) is required")
        if not sp.issparse(self.interaction_matrix):
            raise TypeError("interaction_matrix must be a scipy sparse matrix")

        self.register_buffer('norm_adj_matrix', self._build_norm_adj(self.interaction_matrix).coalesce())
        
        edge_weights = args.get('edge_weights', None)
        self.register_buffer('norm_adj_matrix', 
            self._build_norm_adj_weighted(self.interaction_matrix, edge_weights).coalesce())


    #  Initialization 
    def _init_weights(self, m):
        if isinstance(m, nn.Embedding):
            nn.init.xavier_normal_(m.weight)

    #  Adjacency 
    def _build_norm_adj(self, inter_M: sp.coo_matrix) -> torch.sparse.FloatTensor:
        """Build symmetric normalized adjacency A_hat for user-item bipartite graph."""
        inter_M = inter_M.tocoo()
        A = sp.dok_matrix((self.num_users + self.num_items, self.num_users + self.num_items), dtype=np.float32)
        # user->item (offset items by num_users)
        data_dict = dict(zip(zip(inter_M.row, inter_M.col + self.num_users), [1]*inter_M.nnz))
        # item->user
        data_dict.update(dict(zip(zip(inter_M.col + self.num_users, inter_M.row), [1]*inter_M.nnz)))
        # A._update(data_dict)
        for u, v in data_dict.keys():
            A[u, v] = 1.0

        sum_arr = (A > 0).sum(axis=1)
        deg = np.array(sum_arr.flatten())[0] + 1e-7
        deg_inv_sqrt = np.power(deg, -0.5)
        D = sp.diags(deg_inv_sqrt)
        L = D * A * D  # symmetric norm
        L = sp.coo_matrix(L)
        indices = torch.LongTensor(np.vstack([L.row, L.col]))
        values = torch.FloatTensor(L.data)
        return torch.sparse.FloatTensor(indices, values, torch.Size(L.shape))
    
    def _build_norm_adj_weighted(self, inter_M: sp.coo_matrix, weights=None) -> torch.sparse.FloatTensor:
        """Support weighted edges"""
        inter_M = inter_M.tocoo()
        A = sp.dok_matrix((self.num_users + self.num_items, 
                        self.num_users + self.num_items), dtype=np.float32)
        
        if weights is not None:
            # Use provided weights instead of 1s
            data_dict = dict(zip(zip(inter_M.row, inter_M.col + self.num_users), weights))
            data_dict.update(dict(zip(zip(inter_M.col + self.num_users, inter_M.row), weights)))
        else:
            data_dict = dict(zip(zip(inter_M.row, inter_M.col + self.num_users), [1]*inter_M.nnz))
            data_dict.update(dict(zip(zip(inter_M.col + self.num_users, inter_M.row), [1]*inter_M.nnz)))
        
        # A._update(data_dict)
        for u, v in data_dict.keys():
            A[u, v] = 1.0   

        sum_arr = (A > 0).sum(axis=1)
        deg = np.array(sum_arr.flatten())[0] + 1e-7
        deg_inv_sqrt = np.power(deg, -0.5)
        D = sp.diags(deg_inv_sqrt)
        L = D * A * D  # symmetric norm
        L = sp.coo_matrix(L)
        indices = torch.LongTensor(np.vstack([L.row, L.col]))
        values = torch.FloatTensor(L.data)
        return torch.sparse.FloatTensor(indices, values, torch.Size(L.shape))

    #  Forward Propagation 
    def forward(self) -> Tuple[torch.Tensor, torch.Tensor]:
        all_embeddings = torch.cat([self.embed_user.weight, self.embed_item.weight], dim=0)
        embeddings_list = [all_embeddings]
        for _ in range(self.num_layers):
            all_embeddings = torch.sparse.mm(self.norm_adj_matrix, all_embeddings)
            all_embeddings = self.dropout(all_embeddings)
            embeddings_list.append(all_embeddings)
        # Mean over layers
        final = torch.mean(torch.stack(embeddings_list, dim=1), dim=1)
        user_final, item_final = torch.split(final, [self.num_users, self.num_items])
        return user_final, item_final

    def _bpr_loss(self, pos_scores, neg_scores):
        return -torch.mean(F.logsigmoid(pos_scores - neg_scores))

    def calc_loss(self, batch):
        # ensure model is on correct device
        self.to(self.device)
        # clear stored embeddings before computing
        if self.restore_user_e is not None or self.restore_item_e is not None:
            self.restore_user_e, self.restore_item_e = None, None

        # prepare batch indices
        user = batch[0].to(self.device).long()
        if user.dim() == 0:
            user = user.unsqueeze(0)
        pos_item = batch[1].to(self.device).long()
        if pos_item.dim() == 0:
            pos_item = pos_item.unsqueeze(0)

        # compute embeddings
        embed_user, embed_item = self.forward()
        embed_user = embed_user.to(self.device)
        embed_item = embed_item.to(self.device)

        # positive predictions
        u_emb = embed_user[user]
        p_emb = embed_item[pos_item]
        pos_pred = (u_emb * p_emb).sum(dim=1)

        # ego embeddings for regularization
        u_ego = self.embed_user(user)
        p_ego = self.embed_item(pos_item)

        # compute loss
        neg = batch[2].to(self.device).long()
        neg_emb = embed_item[neg]
        neg_pred = (u_emb * neg_emb).sum(dim=1)
        neg_ego = self.embed_item(neg)
        loss = self._bpr_loss(pos_pred, neg_pred)
        loss += self.reg_1 * (u_ego.norm(p=1) + p_ego.norm(p=1) + neg_ego.norm(p=1))
        loss += self.reg_2 * (u_ego.norm() + p_ego.norm() + neg_ego.norm())

        return loss

    def _ensure_cached_embeddings(self):
        if self.restore_user_e is None or self.restore_item_e is None:
            was_training = self.training
            self.eval()
            with torch.no_grad():
                self.restore_user_e, self.restore_item_e = self.forward()
            if was_training:
                self.train()

    #  Ranking 
    def rank(self, test_loader):
        self._ensure_cached_embeddings()

        rec_ids = torch.tensor([], device=self.device)
        self.eval()
        with torch.no_grad():
            for us in test_loader:
                us = us.to(self.device)
                rank_list = self.full_rank(us)

                rec_ids = torch.cat((rec_ids, rank_list), 0)

        return rec_ids.cpu().numpy().astype(np.int32)
    
    def full_rank(self, u):
        self._ensure_cached_embeddings()

        # ensure CPU indices for CPU embeddings and convert to long type
        if u.device != self.restore_user_e.device:
            u_idx = u.to(self.restore_user_e.device).long()
        else:
            u_idx = u.long()
        user_emb = self.restore_user_e[u_idx]  # (batch_size, dim)
        items_emb = self.restore_item_e  # (num_items, dim)
        # compute scores and top-k
        # scores = torch.matmul(user_emb, items_emb.transpose(1, 0))
        # rank = torch.argsort(scores, descending=True)[:, :self.topk]
        scores = torch.matmul(user_emb, items_emb.t())              # [B, num_items]
        _, rank = torch.topk(scores, k=self.topk, dim=1, largest=True, sorted=True)
        # move to evaluation device
        return rank.to(self.device)
    
    def predict(self, u, i):
        self._ensure_cached_embeddings()
        dev = self.restore_user_e.device

        # convert indices to tensor on same device
        if not torch.is_tensor(u):
            u = torch.tensor(u, device=dev, dtype=torch.long)
        else:
            u = u.to(dev).long()

        if not torch.is_tensor(i):
            i = torch.tensor(i, device=dev, dtype=torch.long)
        else:
            i = i.to(dev).long()

        u_embedding = self.restore_user_e[u]
        i_embedding = self.restore_item_e[i]
        pred = torch.matmul(u_embedding, i_embedding.t())

        return pred.detach().cpu().item()

    def fit(self, train_loader, val_loader, epochs: int = 10):
        opt = optim.Adam(self.parameters(), lr=self.lr)
        self.to(self.device)

        start = 0
        history = {'train_loss': [], 'val_ndcg': [], 'val_recall': [], 'val_precision': [], 'val_auc': []}
        best_ndcg = -np.inf
        patience_counter = 0
        if (Path(self.save_path) / 'last_model.pth').exists() and self.load:
            print("Load model from checkpoint")
            checkpoint = torch.load(Path(self.save_path) / 'last_model.pth', map_location=self.device, weights_only=False)
            self.load_state_dict(checkpoint['model_state_dict'])
            opt.load_state_dict(checkpoint['optimizer_state_dict'])
            start = checkpoint['epoch'] + 1
            history = checkpoint['history']
            best_ndcg = max(history['val_ndcg']) if len(history['val_ndcg']) > 0 else -np.inf   
            self.restore_user_e = None
            self.restore_item_e = None


        for epoch in range(start, epochs+1):
            self.train()
            current_loss = 0.0
            pbar = tqdm(train_loader)
            pbar.set_description(f'[Epoch {epoch:03d}]')
            for batch in pbar:
                opt.zero_grad()
                loss = self.calc_loss(batch)
                if torch.isnan(loss):
                    raise ValueError("NaN loss encountered")
                loss.backward()
                opt.step()
                current_loss += loss.item()

            epoch_loss = current_loss / len(train_loader)
            pbar.set_postfix(loss=epoch_loss)
            history['train_loss'].append(epoch_loss)

            preds = self.rank(val_loader)
            ndcg = NDCG(self.val_ur, preds, self.val_u)
            recall = Recall(self.val_ur, preds, self.val_u)
            precision = Precision(self.val_ur, preds, self.val_u)
            history['val_ndcg'].append(ndcg)
            history['val_recall'].append(recall)
            history['val_precision'].append(precision)
            # if self.true_neg:
            #     if self.true_neg_dict is not None:
            #         auc = AUC_true_neg(self.val_ur, self.restore_user_e[self.val_u].detach().cpu().numpy() @ self.restore_item_e.detach().cpu().numpy().T, self.true_neg_dict, self.val_u)
            #     else:
            #         auc = 0.5
            #     history['val_auc'].append(auc)
            #     print(f"Training - Loss {epoch_loss:.4f} | Validation - NDCG@{self.topk}: {ndcg:.4f}, Recall@{self.topk}: {recall:.4f}, Precision@{self.topk}: {precision:.4f}, AUC@{self.topk}: {auc:.4f}")
            # else:
            print(f"Training - Loss {epoch_loss:.4f} | Validation - NDCG@{self.topk}: {ndcg:.4f}, Recall@{self.topk}: {recall:.4f}, Precision@{self.topk}: {precision:.4f}")
            
            # Save best model
            state = {
                'epoch': epoch,
                'model_state_dict': self.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'history': history
            }
            if ndcg > best_ndcg:
                best_ndcg = ndcg
                torch.save(state, Path(self.save_path) / 'best_model.pth')
                patience_counter = 0
            else:
                patience_counter += 1
                print(f'Patience counter: {patience_counter}/5')
                
            torch.save(state, Path(self.save_path) / 'last_model.pth')
            
            # Early stopping
            if self.early_stop and patience_counter >= 5:
                print('Satisfy early stop mechanism')
                break
            
            self.restore_user_e = None
            self.restore_item_e = None

        return history
    


# ============================================= RF denoiser =============================================
def build_behavior_features(df):
    """
    df: raw_df_train (TRAIN-ONLY)
    Required cols: user_id, item_id, click, like, share, follow, watching_times
    """
    agg = df.groupby(['user_id','item_id']).agg(
        count_click=('click','sum'),
        count_like=('like','sum'),
        count_share=('share','sum'),
        count_follow=('follow','sum'),
        avg_watch_time=('watching_times','mean')
    ).reset_index()

    return agg

def normalize_video_category(x):
    if x == 0:
        return 0
    if x == 1:
        return 1
    if isinstance(x, str):
        x = x.strip()
        if x == '0':
            return 2
        if x == '1':
            return 3
        return 4
    return 4

# ============================================= IMPROVED RF LABELING =============================================

def build_rf_features_v2(raw_df_train):
    """
    IMPROVED label definition for clicks - multi-criteria approach
    Returns X (features), y (label), feature_names
    """
    
    # -------- behavior --------
    beh = build_behavior_features(raw_df_train)
    
    # -------- user meta --------
    user_meta = raw_df_train[['user_id','gender','age']].drop_duplicates()
    
    # -------- item meta --------
    item_meta = raw_df_train[['item_id','video_category']].drop_duplicates()
    item_meta['video_category'] = (
        item_meta['video_category']
        .apply(normalize_video_category)
        .astype(np.int32)
    )
    
    # -------- Calculate user watch time statistics --------
    user_watch_stats = raw_df_train.groupby('user_id')['watching_times'].agg(['mean', 'std']).reset_index()
    user_watch_stats['std'] = user_watch_stats['std'].fillna(0)  # Handle single-interaction users
    
    # -------- merge --------
    feat = beh.merge(user_meta, on='user_id', how='left') \
              .merge(item_meta, on='item_id', how='left') \
              .merge(user_watch_stats, on='user_id', how='left')
    
    feat[['gender','age','video_category','mean','std']] = feat[['gender','age','video_category','mean','std']].fillna(0)
    
    # ===== NEW MULTI-CRITERIA LABEL DEFINITION =====
    
    # Criterion 1: Has explicit engagement (like/share/follow)
    feat['has_engagement'] = (
        (feat['count_like'] > 0) | 
        (feat['count_share'] > 0) | 
        (feat['count_follow'] > 0)
    ).astype(int)
    
    # Criterion 2: High watch time (relative to user's average)
    # Positive if watch time > user_mean + 0.5*user_std
    feat['high_watch'] = (
        feat['avg_watch_time'] > (feat['mean'] + 0.5 * feat['std'])
    ).astype(int)
    
    # Criterion 3: Repeated clicks (user clicked multiple times)
    feat['repeated'] = (feat['count_click'] >= 2).astype(int)
    
    # Criterion 4: Long watch time in absolute terms (> 30 seconds as example)
    feat['long_watch'] = (feat['avg_watch_time'] > 30).astype(int)
    
    # FINAL LABEL: Positive if ANY of the criteria is met
    feat['label'] = (
        (feat['has_engagement'] == 1) | 
        (feat['high_watch'] == 1) | 
        (feat['repeated'] == 1) |
        (feat['long_watch'] == 1)
    ).astype(int)
    
    # Print label statistics
    print(f"\n[RF Label V2 Statistics]")
    print(f"  Total interactions: {len(feat)}")
    print(f"  Positive ratio: {feat['label'].mean():.2%}")
    print(f"  Breakdown:")
    print(f"    - Has engagement (like/share/follow): {feat['has_engagement'].mean():.2%}")
    print(f"    - High watch time (relative): {feat['high_watch'].mean():.2%}")
    print(f"    - Repeated clicks (≥2): {feat['repeated'].mean():.2%}")
    print(f"    - Long watch time (>30s): {feat['long_watch'].mean():.2%}")
    
    y = feat['label'].values
    
    # Drop auxiliary columns for training
    X = feat.drop(columns=['user_id','item_id','label', 'has_engagement', 'high_watch', 'repeated', 'long_watch', 'mean', 'std'])
    
    return feat[['user_id','item_id']], X.values, y, X.columns.tolist()


def train_rf_denoiser_v2(raw_df_train, cf_args):
    """
    Train RF denoiser with IMPROVED label definition
    """
    ui_keys, X, y, feat_names = build_rf_features_v2(raw_df_train)
    
    print(f"\n[Training RF with improved labels]")
    print(f"  Features: {len(feat_names)}")
    print(f"  Samples: {len(X)}")
    print(f"  Positive class: {y.sum()} ({y.mean():.2%})")
    
    rf = RandomForestClassifier(
        n_estimators=cf_args['rf_n_estimators'],
        max_depth=cf_args['rf_max_depth'],
        random_state=cf_args['rf_random_state'],
        n_jobs=-1,
        class_weight='balanced'  # Handle class imbalance
    )
    rf.fit(X, y)
    
    probs = rf.predict_proba(X)[:,1]
    
    res = ui_keys.copy()
    res['prob'] = probs
    res['label'] = y
    
    # Feature importance
    importance = pd.DataFrame({
        'feature': feat_names,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\n[Top 10 Important Features]")
    print(importance.head(10).to_string(index=False))
    
    return rf, res, feat_names


def prepare_training_data(raw_df_train, train_click, cf_args):
    """
    Prepare training data with IMPROVED denoising
    Returns:
      train_pos_df : edges for LightGCN training (ALL clicks, no removal)
      true_neg_df  : true negatives for sampler
      edge_weights : weights for each edge (optional)
    """
    
    if not cf_args.get('denoise', False):
        print("[INFO] Denoise OFF → use click-only training")
        return train_click[['user_id','item_id']], None, None
    
    print("[INFO] Denoise ON → training IMPROVED RF denoiser")
    rf, rf_res, feat_names = train_rf_denoiser_v2(raw_df_train, cf_args)
    
    # Analyze probability distribution
    print(f"\n[RF Probability Distribution]")
    print(f"  Min: {rf_res['prob'].min():.4f}")
    print(f"  25%: {rf_res['prob'].quantile(0.25):.4f}")
    print(f"  50%: {rf_res['prob'].quantile(0.50):.4f}")
    print(f"  75%: {rf_res['prob'].quantile(0.75):.4f}")
    print(f"  Max: {rf_res['prob'].max():.4f}")
    print(f"  Mean: {rf_res['prob'].mean():.4f}")
    
    # Define thresholds
    neg_th = cf_args.get('soft_denoise_neg_th', 0.3)
    
    # True negatives: low probability clicks
    true_neg_df = rf_res[rf_res['prob'] <= neg_th][['user_id','item_id']]
    
    # KEEP ALL CLICKS (no removal!)
    pos_df = train_click[['user_id','item_id']].copy()
    
    # Optional: Return weights if using soft weighting
    if cf_args.get('use_soft_weights', False):
        # Merge RF probs with train_click
        train_click_with_prob = train_click.merge(
            rf_res[['user_id','item_id','prob']],
            on=['user_id','item_id'],
            how='left'
        )
        train_click_with_prob['prob'].fillna(0.5, inplace=True)
        
        # Apply weight transformation
        weights = train_click_with_prob['prob'].values
        
        # Transform weights to avoid very low values
        transform = cf_args.get('weight_transform', 'clip_rescale')
        
        if transform == 'clip_rescale':
            # Clip to [0.3, 1.0] range
            weights = np.clip(weights, 0.3, 1.0)
        elif transform == 'sqrt_shift':
            weights = np.sqrt(weights) * 0.7 + 0.3
            weights = np.clip(weights, 0.3, 1.0)
        elif transform == 'sigmoid':
            weights = 1 / (1 + np.exp(-10 * (weights - 0.5)))
            weights = weights * 0.7 + 0.3
        
        print(f"\n[Edge Weights]")
        print(f"  Transform: {transform}")
        print(f"  Min: {weights.min():.4f}")
        print(f"  Mean: {weights.mean():.4f}")
        print(f"  Max: {weights.max():.4f}")
        print(f"  % weights > 0.5: {(weights > 0.5).mean():.2%}")
    else:
        weights = None
    
    print(f"\n[Final Data Summary]")
    print(f"  Training edges (ALL clicks): {len(pos_df)}")
    print(f"  True negatives (prob ≤ {neg_th}): {len(true_neg_df)}")
    if weights is not None:
        print(f"  Using soft weights: YES")
    else:
        print(f"  Using soft weights: NO (binary graph)")
    
    return pos_df, true_neg_df, weights

def denoise_interactions(rf_res, pos_th=0.6, neg_th=0.4):
    """
    rf_res: output of train_rf_denoiser
    """
    pos_df = rf_res[rf_res['prob'] >= pos_th][['user_id','item_id']]
    true_neg_df = rf_res[rf_res['prob'] <= neg_th][['user_id','item_id']]

    return pos_df.drop_duplicates(), true_neg_df.drop_duplicates()

# ============================================= plot =============================================
def plot_training_curves(history, cf_args):
    os.makedirs(cf_args['plot_path'], exist_ok=True)
    exp_dir = os.path.join(cf_args['plot_path'], cf_args['exp_name'])
    os.makedirs(exp_dir, exist_ok=True)

    epochs = range(1, len(history['train_loss']) + 1)
    %matplotlib inline
    # ----- LOSS -----
    plt.figure()
    plt.plot(epochs, history['train_loss'])
    plt.xlabel('Epoch')
    plt.ylabel('BPR Loss')
    plt.title('Training Loss')
    if cf_args['plot_path']:
        plt.savefig(os.path.join(exp_dir, 'train_loss.png'), dpi=300)
    plt.show()

    # ----- VALIDATION METRICS -----
    if len(history['val_recall']) > 0:
        plt.figure()
        plt.plot(epochs, history['val_recall'], label='Recall')
        plt.plot(epochs, history['val_ndcg'], label='NDCG')
        plt.xlabel('Epoch')
        plt.ylabel('Score')
        plt.title(f'Validation Metrics @K={cf_args["k"]}')
        plt.legend()
        if cf_args['plot_path']:
            plt.savefig(os.path.join(exp_dir, 'val_metrics.png'), dpi=300)
        plt.show()

In [ ]:
# CF Task Pipeline
# 1. Define CF parameters
cf_args = {
    'dataset_path': '../data/data.csv',
    'device': 'cuda',
    'exp_name': 'A5_Full', # A0_Baseline, A1_Rerank, A2_Denoise, A3_TrueNeg, A4_HybridNeg, A5_Full
    
    # Training
    'train_batch_size': 2048,
    'val_batch_size': 64,
    'test_batch_size': 64,

    # Model
    'embedding_dim': 128,
    'lr': 0.005,
    'num_layers': 3,
    'epochs': 30,
    'k': 20,
    'reg_1': 0.0,
    'reg_2': 0.0,

    # Negative sampling
    'sample_method': 'uniform',  # uniform, high-pop, low-pop, true_neg, hybrid
    'sample_ratio': 0.5,
    'num_ng': 4,

    # Data split
    'test_size': 0.1,
    'val_size': 0.1111,

    # RF denoise
    'rf_n_estimators': 300,
    'rf_max_depth': 14,
    'rf_random_state': 42,

    # rerank
    'rerank_topN': 1000,
    'rerank_rf_estimators': 300,
    'rerank_rf_max_depth': 14,
    'rerank_use_pairwise_beh': False,      # tắt beh_0..beh_4
    'rerank_use_engagement': False,        # tắt engagement
    'rerank_use_norm_watch': False,        # tắt norm_watch
    'rerank_longtail_weight': True,        # bật long-tail weight
    'rerank_tail_alpha': 1.0,              # mức mạnh (tăng/giảm tùy)
    'rerank_neg_per_user': 200,            # số negative samples per user for training rerank RF

    # hybrid
    'early_stop': False,    # early stop for LightGCN training
    'true_neg': True,       # use true negatives in sampler
    'denoise': True,        # denoise with RF
    'rerank': True,         # rerank with RF
    'plot': True,           # plot training curves
    'load': False,          # load model from checkpoint
    'load_rf': False,       # load RF model from checkpoint
}
cf_args['save_path'] = os.path.join('./checkpoint/', cf_args['exp_name'])
cf_args['plot_path'] = os.path.join('./plots/', cf_args['exp_name'])
os.makedirs(cf_args['save_path'], exist_ok=True)
os.makedirs(cf_args['plot_path'], exist_ok=True)

# 2. Load and preprocess data
# ===== Load data =====
raw_df = pd.read_csv(cf_args['dataset_path'])
raw_df.fillna(0, inplace=True)

# ===== Fixed-universe mapping =====
user_ids = raw_df['user_id'].unique()
item_ids = raw_df['item_id'].unique()

user_map = {u:i for i,u in enumerate(user_ids)}
item_map = {i:j for j,i in enumerate(item_ids)}

cf_args['user_num'] = len(user_map)
cf_args['item_num'] = len(item_map)
def encode_ui(df):
    df = df.copy()
    # Đảm bảo user_id và item_id là cột, nếu đang là index thì reset về cột
    if 'user_id' not in df.columns and getattr(df.index, 'name', None) == 'user_id':
        df = df.reset_index()  # đưa index user_id thành cột
    if 'item_id' not in df.columns and getattr(df.index, 'name', None) == 'item_id':
        df = df.reset_index()  # đưa index item_id thành cột
    # Thực hiện mapping nếu cột tồn tại
    if 'user_id' in df.columns:
        df['user_id'] = df['user_id'].map(user_map)
    if 'item_id' in df.columns:
        df['item_id'] = df['item_id'].map(item_map)
    
    # Remove NaN (unmapped IDs) and convert to int
    df = df.dropna(subset=[col for col in ['user_id', 'item_id'] if col in df.columns])
    if 'user_id' in df.columns:
        df['user_id'] = df['user_id'].astype(int)
    if 'item_id' in df.columns:
        df['item_id'] = df['item_id'].astype(int)
    
    return df

print(f"Users={cf_args['user_num']}, Items={cf_args['item_num']}")

# 3. Split train/val/test
# ===== Click-positive interactions =====
click_df = (
    raw_df[raw_df.click.isin([1])]
    .drop_duplicates()
    .reset_index(drop=True)
)

# ===== Train / Test =====
train_idx, test_idx = split_test(click_df, cf_args['test_size'])

train_click_raw_full = click_df.iloc[train_idx].reset_index(drop=True)
test_click_raw       = click_df.iloc[test_idx].reset_index(drop=True)

# ===== Train / Val (IMPORTANT: split on FULL train, then slice from FULL train) =====
train_idx2, val_idx = split_validation(train_click_raw_full, cf_args['val_size'])

train_click_raw = train_click_raw_full.iloc[train_idx2].reset_index(drop=True)
val_click_raw   = train_click_raw_full.iloc[val_idx].reset_index(drop=True)

# Encode
train_click = encode_ui(train_click_raw)
val_click = encode_ui(val_click_raw)
test_click = encode_ui(test_click_raw)

train_ur = get_ur(train_click)
val_ur = get_ur(val_click)
test_ur = get_ur(test_click)

cf_args['train_ur'] = train_ur

cf_args['val_ur'] = val_ur

cf_args['test_ur'] = test_ur
cf_args['test_u'] = np.array(list(test_ur.keys()))
cf_args['val_u'] = np.array(list(val_ur.keys()))
# ===================== STRICT TRAIN-ONLY raw_df for feature building =====================
# dùng TRAIN users (raw ids)
train_users_raw = set(train_click_raw['user_id'].values)

# mọi (u,i) thuộc VAL/TEST click đều phải loại khỏi raw_df_train để tránh leakage
holdout_pairs = pd.concat([
    val_click_raw[['user_id','item_id']],
    test_click_raw[['user_id','item_id']]
], axis=0).drop_duplicates()

raw_df_train = raw_df[raw_df['user_id'].isin(train_users_raw)].copy()

# anti-join: drop all rows whose (u,i) is in holdout_pairs
raw_df_train = raw_df_train.merge(
    holdout_pairs.assign(_holdout=1),
    on=['user_id','item_id'],
    how='left'
)
raw_df_train = raw_df_train[raw_df_train['_holdout'].isna()].drop(columns=['_holdout'])

print("[INFO] raw_df_train(strict) rows:", len(raw_df_train))

In [ ]:
# 4. Denoise (if needed)

if cf_args['denoise']:
    rf, rf_res, feat_names = train_rf_denoiser_v2(raw_df_train, cf_args)

    # pos_df from RF denoiser contains raw IDs, need encoding
    pos_df, true_neg_df = denoise_interactions(rf_res, pos_th=0.5, neg_th=0.3)
    print(f"[RF v2] Denoised positives: {len(pos_df)}")
    print(f"[RF v2] True negatives: {len(true_neg_df)}")

    # Validate weight distribution
    print(f"\n[RF v2] Weight distribution:")
    print(f"  Mean: {rf_res['prob'].mean():.4f}")
    print(f"  Median: {rf_res['prob'].median():.4f}")
    print(f"  Std: {rf_res['prob'].std():.4f}")
    print(f"  25%: {rf_res['prob'].quantile(0.25):.4f}")
    print(f"  75%: {rf_res['prob'].quantile(0.75):.4f}")

    pos_df = encode_ui(pos_df)
    if true_neg_df is not None:
        true_neg_df = encode_ui(true_neg_df)
else:
    # train_click is already encoded, use directly without re-encoding
    pos_df = train_click.copy()
    true_neg_df = None

print(f"Final training positives: {len(pos_df)}")

# Validate encoded IDs
print(f"[DEBUG] pos_df after encoding: {len(pos_df)} edges")
print(f"[DEBUG] user_id range: [{pos_df['user_id'].min()}, {pos_df['user_id'].max()}]")
print(f"[DEBUG] item_id range: [{pos_df['item_id'].min()}, {pos_df['item_id'].max()}]")
assert pos_df['user_id'].min() >= 0, "Negative user_id found!"
assert pos_df['item_id'].min() >= 0, "Negative item_id found!"

assert pos_df['user_id'].max() < cf_args['user_num'], f"user_id out of range: {pos_df['user_id'].max()} >= {cf_args['user_num']}"

assert pos_df['item_id'].max() < cf_args['item_num'], f"item_id out of range: {pos_df['item_id'].max()} >= {cf_args['item_num']}"

# build graph
graph_df = pos_df.copy()

print(f"Number of users in graph: {graph_df['user_id'].nunique()}")
print(f"Number of items in graph: {graph_df['item_id'].nunique()}")

graph_df['click'] = 1
cf_args['interaction_matrix'] = get_inter_matrix(graph_df, cf_args)
cf_args['true_neg_dict'] = HybridNegativeSampler._build_true_neg_dict(true_neg_df) if cf_args['true_neg'] else None


In [ ]:
# sampler
sampler = HybridNegativeSampler(cf_args)
triples = sampler.sampling(
    train_edges_df=pos_df,
    train_ur=train_ur,
    true_neg_df=true_neg_df if cf_args['true_neg'] else None
)
# train
train_ds = BasicDataset(triples)
train_loader = get_train_loader(train_ds, cf_args)
val_loader = get_val_loader(Cf_valDataset(cf_args['val_u']), cf_args)
model = LightGCN(cf_args)
history = model.fit(train_loader, val_loader, epochs=cf_args['epochs'])

# Create necessary directories
if cf_args['plot']:
    plot_training_curves(history, cf_args)

In [ ]:
def Recall_at_k(test_ur, pred_ur, test_u, k=20):
    res = []
    for idx in range(len(test_u)):
        u = test_u[idx]
        gt = test_ur[u]
        pred = pred_ur[idx][:k]  # IMPORTANT: cut topK
        rec = np.isin(pred, list(gt)).sum() / len(gt)
        res.append(rec)
    return float(np.mean(res))

def NDCG_at_k(test_ur, pred_ur, test_u, k=20):
    res = []
    for idx in range(len(test_u)):
        u = test_u[idx]
        gt = list(test_ur[u])
        pred = pred_ur[idx][:k]  # IMPORTANT: cut topK
        res.append(getNDCG(pred, gt))
    return float(np.mean(res))


def get_topN_candidates(model, users, cf_args):
    """
    Returns: np.ndarray [num_users, topN]
    """
    model.topk = cf_args.get('rerank_topN', 20)
    loader = get_test_loader(Cf_valDataset(users), cf_args)
    return model.rank(loader)

def build_behavior_features_ui(raw_df_train):
    return raw_df_train.groupby(['user_id','item_id']).agg(
        cnt_click=('click','sum'),
        cnt_like=('like','sum'),
        cnt_share=('share','sum'),
        cnt_follow=('follow','sum'),
        avg_watch=('watching_times','mean')
    ).reset_index()

def get_user_item_meta(raw_df):
    user_meta = raw_df[['user_id','gender','age']].drop_duplicates()
    item_meta = raw_df[['item_id','video_category']].drop_duplicates()
    return user_meta, item_meta

def _safe_mode(arr):
    """Fast-ish mode for 1D numpy array of ints; returns -1 if empty."""
    if arr is None or len(arr) == 0:
        return -1
    vals, cnt = np.unique(arr, return_counts=True)
    return int(vals[np.argmax(cnt)])

def _build_user_item_dicts(raw_df_train, raw_df_all):
    """
    Build all lookup dicts in encoded id space:
    - beh_dict[(u,i)] -> [cnt_click,cnt_like,cnt_share,cnt_follow,avg_watch]
    - user_meta_dict[u] -> (gender, age)
    - item_cat_dict[i] -> category
    - user_stats_dict[u] -> [activity, avg_watch, like_rate, share_rate]
    - item_stats_dict[i] -> [pop, avg_watch, like_rate, share_rate]
    - user_pref_cat_dict[u] -> most frequent video_category in user's history (encoded space)
    - item_pop_percentile_dict[i] -> percentile in [0,1]
    """
    # ---- behavior (u,i) ----
    beh = raw_df_train.groupby(['user_id','item_id']).agg(
        cnt_click=('click','sum'),
        cnt_like=('like','sum'),
        cnt_share=('share','sum'),
        cnt_follow=('follow','sum'),
        avg_watch=('watching_times','mean')
    ).reset_index()
    beh = encode_ui(beh)
    beh_dict = {(int(u), int(i)): [float(a), float(b), float(c), float(d), float(e)]
                for u, i, a, b, c, d, e in beh.itertuples(index=False)}

    # ---- meta ----
    user_meta = raw_df_all[['user_id','gender','age']].drop_duplicates()
    item_meta = raw_df_all[['item_id','video_category']].drop_duplicates()
    user_meta = encode_ui(user_meta)
    item_meta = encode_ui(item_meta)

    user_meta_dict = {int(u): (float(g), float(a))
                      for u, g, a in user_meta.itertuples(index=False)}
    item_cat_dict = {int(i): float(cat)
                     for i, cat in item_meta.itertuples(index=False)}

    # ---- user stats ----
    # NOTE: like_rate formula in your code is odd; keeping it but making it safe.
    user_stats = raw_df_train.groupby('user_id').agg(
        user_activity=('item_id', 'nunique'),
        user_avg_watch=('watching_times', 'mean'),
        user_like_sum=('like', 'sum'),
        user_cnt=('like', 'size'),
        user_share_rate=('share', 'mean')
    ).reset_index()
    user_stats['user_like_rate'] = user_stats['user_like_sum'] / (user_stats['user_cnt'] + 1e-9)
    user_stats = user_stats[['user_id','user_activity','user_avg_watch','user_like_rate','user_share_rate']]
    user_stats = encode_ui(user_stats)
    user_stats_dict = {int(u): [float(a), float(w), float(lr), float(sr)]
                       for u, a, w, lr, sr in user_stats.itertuples(index=False)}

    # ---- item stats ----
    item_stats = raw_df_train.groupby('item_id').agg(
        item_popularity=('user_id', 'nunique'),
        item_avg_watch=('watching_times', 'mean'),
        item_like_rate=('like', 'mean'),
        item_share_rate=('share', 'mean'),
    ).reset_index()
    item_stats = encode_ui(item_stats)
    item_stats_dict = {int(i): [float(p), float(w), float(lr), float(sr)]
                       for i, p, w, lr, sr in item_stats.itertuples(index=False)}

    # ---- user preferred category (compute ONCE, encoded space, no reverse_map, no df filter per row) ----
    # Build encoded interactions (u,i), then map i -> category, then mode per user
    ui = raw_df_train[['user_id','item_id']].drop_duplicates()
    ui = encode_ui(ui)
    # Map item->cat
    cats = ui['item_id'].map(lambda x: item_cat_dict.get(int(x), -1.0)).astype(np.float32)
    ui_cat = pd.DataFrame({'user_id': ui['user_id'].astype(int).values, 'cat': cats.values})
    # mode per user
    user_pref_cat_dict = {}
    for u, grp in ui_cat.groupby('user_id'):
        user_pref_cat_dict[int(u)] = float(_safe_mode(grp['cat'].values))

    # ---- item popularity percentile (compute ONCE) ----
    items = np.array(list(item_stats_dict.keys()), dtype=np.int32)
    if len(items) > 1:
        pops = np.array([item_stats_dict[int(i)][0] for i in items], dtype=np.float32)
        order = np.argsort(pops, kind='mergesort')
        ranks = np.empty_like(order, dtype=np.int32)
        ranks[order] = np.arange(len(items), dtype=np.int32)
        denom = float(len(items) - 1)
        pct = (ranks.astype(np.float32) / denom)
        item_pop_percentile_dict = {int(i): float(p) for i, p in zip(items, pct)}
    elif len(items) == 1:
        item_pop_percentile_dict = {int(items[0]): 0.0}
    else:
        item_pop_percentile_dict = {}

    return (beh_dict, user_meta_dict, item_cat_dict,
            user_stats_dict, item_stats_dict,
            user_pref_cat_dict, item_pop_percentile_dict)

def build_rerank_features_v3_memmap(
    candidates,
    model,
    users,
    raw_df_train,
    raw_df_all,
    checkpoint_dir='./checkpoint/rerank_features_v3',
    checkpoint_every_users=500,
    load_from_checkpoint=False,
    use_pairwise_beh=False,      # NEW
    use_engagement=False,        # NEW
    use_norm_watch=False         # NEW
):

    """
    Output:
      - X_mm: np.memmap [num_users*topN, feat_dim] float32
      - u_keys_mm: np.memmap [num_users*topN] int32
      - i_keys_mm: np.memmap [num_users*topN] int32
      - meta: dict (paths, shapes, feat_dim, topN, last_user_idx)
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    meta_path = os.path.join(checkpoint_dir, 'rerank_meta.pkl')
    ckpt_path = os.path.join(checkpoint_dir, 'rerank_checkpoint.pkl')

    use_pairwise_beh = cf_args.get('rerank_use_pairwise_beh', False)
    users = np.asarray(users, dtype=np.int32)
    topN = int(candidates.shape[1])
    num_users = int(len(users))
    total_rows = num_users * topN

    # embeddings ONCE
    U_emb, I_emb = model.forward()
    U_emb = U_emb.detach().cpu().numpy().astype(np.float32, copy=False)
    I_emb = I_emb.detach().cpu().numpy().astype(np.float32, copy=False)
    emb_dim = int(U_emb.shape[1])

    # lookup dicts ONCE
    (beh_dict, user_meta_dict, item_cat_dict,
     user_stats_dict, item_stats_dict,
     user_pref_cat_dict, item_pop_pct_dict) = _build_user_item_dicts(raw_df_train, raw_df_all)

    feat_dim = 2 * emb_dim + 2 + 5 + 2 + 1 + 4 + 4 + 5

    # allocate or resume memmaps
    X_path = os.path.join(checkpoint_dir, f'X_float32_{total_rows}x{feat_dim}.dat')
    ukey_path = os.path.join(checkpoint_dir, f'ukey_int32_{total_rows}.dat')
    ikey_path = os.path.join(checkpoint_dir, f'ikey_int32_{total_rows}.dat')

    if os.path.exists(ckpt_path) and os.path.exists(X_path) and os.path.exists(ukey_path) and os.path.exists(ikey_path) and load_from_checkpoint:
        with open(ckpt_path, 'rb') as f:
            ckpt = pickle.load(f)
        start_user_idx = int(ckpt.get('last_user_idx', -1)) + 1
        print(f"[INFO] Resuming from user {start_user_idx}/{num_users}")
    else:
        start_user_idx = 0
        # fresh: create empty files by opening memmap in w+ mode
        _ = np.memmap(X_path, dtype=np.float32, mode='w+', shape=(total_rows, feat_dim))
        _ = np.memmap(ukey_path, dtype=np.int32, mode='w+', shape=(total_rows,))
        _ = np.memmap(ikey_path, dtype=np.int32, mode='w+', shape=(total_rows,))
        del _

    X_mm = np.memmap(X_path, dtype=np.float32, mode='r+', shape=(total_rows, feat_dim))
    u_keys_mm = np.memmap(ukey_path, dtype=np.int32, mode='r+', shape=(total_rows,))
    i_keys_mm = np.memmap(ikey_path, dtype=np.int32, mode='r+', shape=(total_rows,))

    def _write_checkpoint(last_user_idx):
        with open(ckpt_path, 'wb') as f:
            pickle.dump({'last_user_idx': int(last_user_idx)}, f)
        # also save meta for easier loading elsewhere
        meta = {
            'X_path': X_path, 'ukey_path': ukey_path, 'ikey_path': ikey_path,
            'total_rows': total_rows, 'feat_dim': feat_dim, 'topN': topN, 'num_users': num_users,
            'emb_dim': emb_dim
        }
        with open(meta_path, 'wb') as f:
            pickle.dump(meta, f)
        print(f"[CHECKPOINT] Saved at user {last_user_idx + 1}/{num_users}")

    # main loop (fast lookups)
    for ui in tqdm(range(start_user_idx, num_users), desc="building rerank features (v3)"):
        u = int(users[ui])
        row0 = ui * topN
        cands_u = candidates[ui]

        # Pre-fetch user parts (avoid repeated dict calls)
        u_emb = U_emb[u]
        ug, ua = user_meta_dict.get(u, (0.0, 0.0))
        u_stats = user_stats_dict.get(u, [0.0, 0.0, 0.0, 0.0])
        u_avg_watch = float(u_stats[1])
        u_pref_cat = float(user_pref_cat_dict.get(u, -1.0))
        u_norm = float(np.linalg.norm(u_emb) + 1e-6)

        for r in range(topN):
            i = int(cands_u[r])
            row = row0 + r

            i_emb = I_emb[i]
            dot = float(np.dot(u_emb, i_emb))
            i_norm = float(np.linalg.norm(i_emb) + 1e-6)

            beh5 = beh_dict.get((u, i), [0.0, 0.0, 0.0, 0.0, 0.0])
            i_stats = item_stats_dict.get(i, [0.0, 0.0, 0.0, 0.0])

            if not use_pairwise_beh:
                beh5 = [0.0, 0.0, 0.0, 0.0, 0.0]

            item_cat = float(item_cat_dict.get(i, -1.0))
            cat_match = 1.0 if (u_pref_cat != -1.0 and item_cat != -1.0 and u_pref_cat == item_cat) else 0.0

            engagement = (beh5[0] * 1.0 + beh5[1] * 2.0 + beh5[2] * 3.0 + beh5[3] * 5.0 + beh5[4] * 0.1)
            norm_watch = float(beh5[4] / (u_avg_watch + 1e-6))
            cosine_sim = float(dot / (u_norm * i_norm))
            pop_pct = float(item_pop_pct_dict.get(i, 0.0))

            if not use_engagement:
                engagement = 0.0
            if not use_norm_watch:
                norm_watch = 0.0

            # fill feature vector (no python list append spam)
            # layout matches your v2 names
            off = 0
            X_mm[row, off:off+emb_dim] = u_emb; off += emb_dim
            X_mm[row, off:off+emb_dim] = i_emb; off += emb_dim
            X_mm[row, off] = dot; off += 1
            X_mm[row, off] = float(r); off += 1
            X_mm[row, off:off+5] = np.asarray(beh5, dtype=np.float32); off += 5
            X_mm[row, off] = ug; off += 1
            X_mm[row, off] = ua; off += 1
            X_mm[row, off] = item_cat; off += 1
            X_mm[row, off:off+4] = np.asarray(u_stats, dtype=np.float32); off += 4
            X_mm[row, off:off+4] = np.asarray(i_stats, dtype=np.float32); off += 4
            X_mm[row, off] = float(engagement); off += 1
            X_mm[row, off] = float(norm_watch); off += 1
            X_mm[row, off] = float(cat_match); off += 1
            X_mm[row, off] = float(cosine_sim); off += 1
            X_mm[row, off] = float(pop_pct); off += 1

            u_keys_mm[row] = u
            i_keys_mm[row] = i

        if (ui + 1) % int(checkpoint_every_users) == 0:
            X_mm.flush(); u_keys_mm.flush(); i_keys_mm.flush()
            _write_checkpoint(ui)

    # final flush + meta
    X_mm.flush(); u_keys_mm.flush(); i_keys_mm.flush()
    _write_checkpoint(num_users - 1)

    meta = {
        'X_path': X_path, 'ukey_path': ukey_path, 'ikey_path': ikey_path,
        'total_rows': total_rows, 'feat_dim': feat_dim, 'topN': topN, 'num_users': num_users,
        'emb_dim': emb_dim
    }
    return X_mm, u_keys_mm, i_keys_mm, meta

def load_rerank_memmap(meta_path):
    """Load X/u_keys/i_keys memmaps from meta.pkl."""
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    X = np.memmap(meta['X_path'], dtype=np.float32, mode='r', shape=(meta['total_rows'], meta['feat_dim']))
    u = np.memmap(meta['ukey_path'], dtype=np.int32, mode='r', shape=(meta['total_rows'],))
    i = np.memmap(meta['ikey_path'], dtype=np.int32, mode='r', shape=(meta['total_rows'],))
    return X, u, i, meta

def train_reranker_rf_v3(
    model,
    label_users,          # val users (encoded)
    label_ur,             # val_ur (encoded)
    true_neg_dict,        # true_neg_dict (encoded): {u: list/set neg items}
    raw_df_train,         # strict train-only raw df (raw ids)
    raw_df_all,
    cf_args,
    seen_ur=None,         # train_ur to filter seen items
    neg_per_user=200,     # cap true-neg per user (hard negatives are early in candidate list)
):
    # 1) candidates for label_users (ensure cf_args['rerank_topN'] already set, e.g. 1000)
    candidates = get_topN_candidates(model, label_users, cf_args)  # [U, topN]
    U, topN = candidates.shape
    print(f"[INFO] candidates shape: {candidates.shape}")

    # 2) build features (memmap)
    X_mm, u_mm, i_mm, meta = build_rerank_features_v3_memmap(
        candidates=candidates,
        model=model,
        users=label_users,
        raw_df_train=raw_df_train,
        raw_df_all=raw_df_all,
        checkpoint_dir=os.path.join(cf_args.get('save_path', './checkpoint'), 'rerank_train_v3_true_neg'),
        checkpoint_every_users=cf_args.get('rerank_checkpoint_batch', 500),
        load_from_checkpoint=cf_args.get('load_rf', False),
        use_pairwise_beh=cf_args.get('rerank_use_pairwise_beh', False),
        use_engagement=cf_args.get('rerank_use_engagement', False),
        use_norm_watch=cf_args.get('rerank_use_norm_watch', False),
    )

    # Expect contiguous rows per user: total_rows == U*topN
    assert meta['total_rows'] == U * topN, f"Unexpected total_rows={meta['total_rows']} vs {U}*{topN}"

    # 3) O(1) membership sets
    pos_set  = {int(u): set(map(int, items)) for u, items in label_ur.items()}
    tn_set   = {int(u): set(map(int, items)) for u, items in true_neg_dict.items()}
    seen_set = {int(u): set(map(int, items)) for u, items in (seen_ur or {}).items()}

    # 4) build keep mask + y using (pos) and (true_neg ∩ candidates) only
    y = np.zeros((meta['total_rows'],), dtype=np.int8)
    keep = np.zeros((meta['total_rows'],), dtype=bool)
    neg_cnt = defaultdict(int)

    for idx in range(meta['total_rows']):
        u = int(u_mm[idx]); i = int(i_mm[idx])

        # drop seen train items (avoid trivial repeats)
        if i in seen_set.get(u, set()):
            continue

        # positive: held-out VAL positives
        if i in pos_set.get(u, set()):
            y[idx] = 1
            keep[idx] = True
            continue

        # negative: only true negatives (exposed-but-ignored), capped per user
        if i in tn_set.get(u, set()):
            if neg_cnt[u] < neg_per_user:
                y[idx] = 0
                keep[idx] = True
                neg_cnt[u] += 1

    X_train = X_mm[keep]
    y_train = y[keep]
    kept_u = u_mm[keep].astype(np.int32)
    kept_i = i_mm[keep].astype(np.int32)

    print(f"[INFO] Train rows kept: {len(y_train)} | pos={int(y_train.sum())} | neg={int((y_train==0).sum())}")

    # 5) RF model
    rf = RandomForestClassifier(
        n_estimators=cf_args.get('rerank_rf_estimators', 400),
        max_depth=cf_args.get('rerank_rf_max_depth', 16),
        min_samples_leaf=cf_args.get('rerank_rf_min_leaf', 1),
        random_state=cf_args.get('rerank_rf_random_state', 42),
        n_jobs=-1,
        class_weight=cf_args.get('rerank_class_weight', "balanced"),  # recommended with imbalance
    )

    # 6) Long-tail aware weights (positives only)
    sample_weight = None
    if cf_args.get('rerank_longtail_weight', True):
        # item popularity from STRICT TRAIN ONLY
        item_pop = raw_df_train.groupby('item_id')['user_id'].nunique().reset_index(name='pop')
        item_pop = encode_ui(item_pop)  # -> encoded item_id

        pop_arr = np.ones((cf_args['item_num'],), dtype=np.float32)
        for it, p in item_pop[['item_id', 'pop']].itertuples(index=False):
            it = int(it)
            if 0 <= it < len(pop_arr):
                pop_arr[it] = float(p)

        pop_row = pop_arr[kept_i]  # IMPORTANT: only kept rows
        sample_weight = np.ones((len(y_train),), dtype=np.float32)

        pos_mask = (y_train == 1)
        alpha = float(cf_args.get('rerank_tail_alpha', 1.0))
        # upweight tail positives: 1/(log(1+pop)^alpha)
        sample_weight[pos_mask] = 1.0 / (np.log1p(pop_row[pos_mask]) ** alpha + 1e-6)

        # Note: sklearn will multiply class_weight with sample_weight if both provided :contentReference[oaicite:2]{index=2}
        rf.fit(X_train, y_train, sample_weight=sample_weight)
    else:
        rf.fit(X_train, y_train)

    # 7) Save
    rf_path = os.path.join(cf_args.get('save_path', './checkpoint'), 'rf_reranker_true_neg.pkl')
    joblib.dump(rf, rf_path)
    print(f"[INFO] Random Forest reranker saved to {rf_path}")

    return rf

def rerank_candidates_v3(
    model,
    rf,
    test_users,
    raw_df_train,
    raw_df_all,
    cf_args
):
    save_path = cf_args.get('save_path', './checkpoint')
    os.makedirs(save_path, exist_ok=True)

    cand_path = os.path.join(save_path, 'test_candidates_v3.npy')
    meta_path = os.path.join(save_path, 'rerank_test_v3', 'rerank_meta.pkl')
    test_dir = os.path.join(save_path, 'rerank_test_v3')

    if os.path.exists(cand_path) and os.path.exists(meta_path):
        print(f"[INFO] Loading cached candidates + memmap features")
        candidates = np.load(cand_path, allow_pickle=False)
        X_mm, u_mm, i_mm, meta = load_rerank_memmap(meta_path)
    else:
        print(f"[INFO] Building candidates + memmap features (first time)")
        candidates = get_topN_candidates(model, test_users, cf_args)
        np.save(cand_path, candidates, allow_pickle=False)

        X_mm, u_mm, i_mm, meta = build_rerank_features_v3_memmap(
            candidates=candidates,
            model=model,
            users=test_users,
            raw_df_train=raw_df_train,
            raw_df_all=raw_df_all,
            checkpoint_dir=test_dir,
            checkpoint_every_users=cf_args.get('rerank_checkpoint_batch', 500),
            load_from_checkpoint=cf_args.get('load_rf', False),
            use_pairwise_beh=cf_args.get('rerank_use_pairwise_beh', False),
            use_engagement=cf_args.get('rerank_use_engagement', False),
            use_norm_watch=cf_args.get('rerank_use_norm_watch', False),
        )

    # Predict
    scores = rf.predict_proba(X_mm)[:, 1]

    # Rerank per user
    topN = int(meta['topN'])
    num_users = int(meta['num_users'])

    reranked = np.empty_like(candidates)
    for ui in tqdm(range(num_users), desc="reranking (v3)"):
        row0 = ui * topN
        s = scores[row0:row0+topN]
        order = np.argsort(-s)
        reranked[ui] = candidates[ui][order]

    return reranked

def save_results(history, test_results, cf_args):
    """
    Save training history and test results to JSON file.
    Only saves JSON-serializable configuration parameters.
    """
    exp_dir =cf_args['save_path']
    os.makedirs(exp_dir, exist_ok=True)

    # Select only serializable config fields
    config_to_save = {
        'exp_name': cf_args['exp_name'],
        'embedding_dim': cf_args['embedding_dim'],
        'lr': cf_args['lr'],
        'num_layers': cf_args['num_layers'],
        'epochs': cf_args['epochs'],
        'k': cf_args['k'],
        'reg_1': cf_args['reg_1'],
        'reg_2': cf_args['reg_2'],
        'train_batch_size': cf_args['train_batch_size'],
        'val_batch_size': cf_args['val_batch_size'],
        'test_batch_size': cf_args['test_batch_size'],
        'sample_method': cf_args.get('sample_method', cf_args.get('sampler_method', 'uniform')),
        'sample_ratio': cf_args['sample_ratio'],
        'num_ng': cf_args['num_ng'],
        'test_size': cf_args['test_size'],
        'val_size': cf_args['val_size'],
        'denoise': cf_args['denoise'],
        'true_neg': cf_args['true_neg'],
        'rerank': cf_args['rerank'],
        'user_num': cf_args['user_num'],
        'item_num': cf_args['item_num'],
    }

    result = {
        'config': config_to_save,
        'final_train_loss': float(history['train_loss'][-1]) if history['train_loss'] else None,
        'best_val_recall': float(max(history['val_recall'])) if history['val_recall'] else None,
        'best_val_ndcg': float(max(history['val_ndcg'])) if history['val_ndcg'] else None,
        'test_metrics': {k: float(v) for k, v in test_results.items()}
    }

    with open(os.path.join(exp_dir, 'results.json'), 'w') as f:
        json.dump(result, f, indent=2)

    print(f"[INFO] Results saved to {exp_dir}")


In [ ]:
if cf_args['rerank']:
    rf_path = os.path.join('./checkpoint/', cf_args['exp_name'], 'rf_reranker.pkl')

    if os.path.exists(rf_path) and cf_args.get('load_rf', False):
        print("[INFO] Loading existing Random Forest reranker model")
        rf_reranker = joblib.load(rf_path)
    else:
        # Train reranker on VAL (label_users=val_u, label_ur=val_ur)
        rf_reranker = train_reranker_rf_v3(
            model,
            label_users=cf_args['val_u'],   # encoded
            label_ur=val_ur,                # encoded
            true_neg_dict=cf_args['true_neg_dict'],  # encoded
            raw_df_train=raw_df_train,      # strict train-only raw ids
            raw_df_all=raw_df,
            cf_args=cf_args, 
            seen_ur=train_ur,               # filter train seen
            neg_per_user=cf_args.get('rerank_neg_per_user', 50)

        )

    # Rerank candidates for TEST users
    preds = rerank_candidates_v3(
        model, rf_reranker,
        test_users=cf_args['test_u'],
        raw_df_train=raw_df_train,
        raw_df_all=raw_df,
        cf_args=cf_args
    )
else:
    preds = get_topN_candidates(model, cf_args['test_u'], cf_args)

recall = Recall_at_k(test_ur, preds, cf_args['test_u'], k=cf_args['k'])
ndcg = NDCG_at_k(test_ur, preds, cf_args['test_u'], k=cf_args['k'])
print (f"Test Recall@{cf_args['k']}: {recall:.4f}, NDCG@{cf_args['k']}: {ndcg:.4f}")

In [ ]:

test_results = {
    'Recall@20': recall,
    'NDCG@20': ndcg,
}
save_results(history, test_results, cf_args)
